# LAB 3: Busca Híbrida em Corpus Jurídico
## Avaliação Comparativa de Estratégias de Busca

**Objetivo:** Avaliar a efetividade de diferentes estratégias de busca (BM25, KNN, Hybrid RRF e Hybrid MinMax) em um corpus jurídico, utilizando métricas de recuperação de informação (MRR@10, Recall@10, NDCG@10).

**Pré-requisitos:** LAB1 (Indexação) e LAB2 (Embeddings) já executados

**Escopo:** Análise comparativa com 15 queries de avaliação e cálculo de métricas do zero em Python puro.

## Instalação de Dependências

In [ ]:
!pip install opensearch-py transformers torch sentence-transformers numpy pandas matplotlib seaborn scikit-learn -q

## Importações

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from opensearchpy import OpenSearch
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

print("Importações realizadas com sucesso.")

## 1. Fundamentação Teórica: Métricas de Avaliação

### 1.1 Mean Reciprocal Rank (MRR@10)

Mede o ranking da primeira resposta relevante encontrada:

$$\\text{MRR}@K = \\frac{1}{|Q|} \\sum_{i=1}^{|Q|} \\frac{1}{\\text{rank}_i}$$

Onde rank_i é a posição do primeiro documento relevante. MRR varia de 0 a 1.

### 1.2 Recall@K

$$\\text{Recall}@K = \\frac{|\\text{Relevantes} \\cap \\text{Top-K}|}{|\\text{Relevantes}|}$$

### 1.3 Normalized Discounted Cumulative Gain (NDCG@K)

$$\\text{DCG}@K = \\sum_{i=1}^{K} \\frac{2^{rel_i} - 1}{\\log_2(i+1)}$$

## 2. Conexão com OpenSearch

In [ ]:
try:
    client = OpenSearch(
        hosts=[{'host': 'localhost', 'port': 9200}],
        http_auth=('admin', 'admin'),
        use_ssl=False,
        verify_certs=False,
        ssl_show_warn=False
    )
    info = client.info()
    print(f"Conectado ao OpenSearch: {info['version']['number']}")
except Exception as e:
    print(f"Aviso: OpenSearch não disponível ({e})")
    client = None

## 3. Carregar Modelo de Embeddings

In [ ]:
try:
    model = SentenceTransformer('BAAI/bge-m3')
    print(f"Modelo BGE-M3 carregado: dimensionalidade {model.get_sentence_embedding_dimension()}")
except Exception as e:
    print(f"Erro ao carregar modelo: {e}")
    model = None

## 4. Funções de Cálculo de Métricas

In [ ]:
def calc_mrr(results, relevant_docs, k=10):
    """Calcula Mean Reciprocal Rank para uma query."""
    results_k = results[:k]
    for rank, doc_id in enumerate(results_k, 1):
        if doc_id in relevant_docs:
            return 1.0 / rank
    return 0.0


def calc_recall_at_k(results, relevant_docs, k=10):
    """Calcula Recall@K para uma query."""
    if len(relevant_docs) == 0:
        return 0.0
    results_k = results[:k]
    relevant_found = len(set(results_k) & relevant_docs)
    return relevant_found / len(relevant_docs)


def calc_ndcg_at_k(results, relevant_docs, k=10):
    """Calcula Normalized Discounted Cumulative Gain@K para uma query."""
    dcg = 0.0
    results_k = results[:k]
    for i, doc_id in enumerate(results_k, 1):
        rel = 1 if doc_id in relevant_docs else 0
        dcg += (2**rel - 1) / np.log2(i + 1)
    
    idcg = 0.0
    num_relevant = min(len(relevant_docs), k)
    for i in range(1, num_relevant + 1):
        idcg += (2**1 - 1) / np.log2(i + 1)
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg


print("Funções de métricas definidas.")

## 5. Criar Queries de Avaliação

In [ ]:
queries_avaliacao = [
    {"query_id": "q1", "texto": "prisão preventiva fundamentação jurídica", "documentos_relevantes": ["doc_pp_001", "doc_pp_005", "doc_cp_120"]},
    {"query_id": "q2", "texto": "habeas corpus direito de liberdade", "documentos_relevantes": ["doc_hc_001", "doc_cf_005", "doc_hc_002"]},
    {"query_id": "q3", "texto": "tráfico de drogas pena mínima", "documentos_relevantes": ["doc_td_001", "doc_cp_015", "doc_td_003"]},
    {"query_id": "q4", "texto": "direito ambiental responsabilidade civil", "documentos_relevantes": ["doc_da_001", "doc_cc_050", "doc_da_002"]},
    {"query_id": "q5", "texto": "propriedade intelectual marca registrada", "documentos_relevantes": ["doc_pi_001", "doc_pi_002", "doc_lei_marca"]},
    {"query_id": "q6", "texto": "contrato civil obrigações partes", "documentos_relevantes": ["doc_cc_001", "doc_cc_100", "doc_cc_002"]},
    {"query_id": "q7", "texto": "homicídio qualificado agravantes", "documentos_relevantes": ["doc_hom_001", "doc_cp_121", "doc_hom_002"]},
    {"query_id": "q8", "texto": "direito do consumidor vício produto", "documentos_relevantes": ["doc_dc_001", "doc_cdc_001", "doc_dc_002"]},
    {"query_id": "q9", "texto": "falência insolvência liquidação patrimônio", "documentos_relevantes": ["doc_fal_001", "doc_fal_002", "doc_lf_001"]},
    {"query_id": "q10", "texto": "direito do trabalho rescisão contrato", "documentos_relevantes": ["doc_dt_001", "doc_clt_001", "doc_dt_002"]},
    {"query_id": "q11", "texto": "sucessão hereditária inventário partilha", "documentos_relevantes": ["doc_suc_001", "doc_cc_1920", "doc_suc_002"]},
    {"query_id": "q12", "texto": "processo civil jurisdição competência foro", "documentos_relevantes": ["doc_pc_001", "doc_cpc_001", "doc_pc_002"]},
    {"query_id": "q13", "texto": "execução fiscal obrigação tributária", "documentos_relevantes": ["doc_ef_001", "doc_ctf_001", "doc_ef_002"]},
    {"query_id": "q14", "texto": "direito penal dolo culpa imprudência", "documentos_relevantes": ["doc_dp_001", "doc_cp_001", "doc_dp_002"]},
    {"query_id": "q15", "texto": "família casamento divórcio pensão alimentos", "documentos_relevantes": ["doc_fam_001", "doc_cc_1500", "doc_fam_002"]}
]

print(f"Carregadas {len(queries_avaliacao)} queries de avaliação")

## 6. Funções de Busca por Modo

In [ ]:
def search_bm25(client, query, k=10, index_name='corpus_juridico'):
    """Busca BM25 (lexical full-text)."""
    if client is None:
        return [f'doc_mock_{i}' for i in range(k)]
    try:
        search_body = {"size": k, "query": {"match": {"texto": {"query": query, "operator": "or"}}}}
        response = client.search(index=index_name, body=search_body)
        return [hit['_id'] for hit in response['hits']['hits']]
    except Exception as e:
        print(f"Erro em BM25: {e}")
        return []


def search_knn(client, query, model, k=10, index_name='corpus_juridico'):
    """Busca KNN (similarity search com embeddings)."""
    if client is None or model is None:
        return [f'doc_mock_{i}' for i in range(k)]
    try:
        query_embedding = model.encode(query).tolist()
        search_body = {"size": k, "query": {"knn": {"embedding": {"vector": query_embedding, "k": k}}}}
        response = client.search(index=index_name, body=search_body)
        return [hit['_id'] for hit in response['hits']['hits']]
    except Exception as e:
        print(f"Erro em KNN: {e}")
        return []


def search_hybrid_rrf(client, query, model, k=10, index_name='corpus_juridico'):
    """Busca Híbrida com Reciprocal Rank Fusion (RRF)."""
    if client is None or model is None:
        return [f'doc_mock_{i}' for i in range(k)]
    try:
        bm25_results = search_bm25(client, query, k=100, index_name=index_name)
        knn_results = search_knn(client, query, model, k=100, index_name=index_name)
        
        rrf_scores = {}
        for rank, doc_id in enumerate(bm25_results, 1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1.0 / (60 + rank)
        for rank, doc_id in enumerate(knn_results, 1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1.0 / (60 + rank)
        
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return [doc_id for doc_id, _ in sorted_docs[:k]]
    except Exception as e:
        print(f"Erro em Hybrid RRF: {e}")
        return []


def search_hybrid_minmax(client, query, model, k=10, index_name='corpus_juridico', alpha=0.5):
    """Busca Híbrida com Min-Max Normalization."""
    if client is None or model is None:
        return [f'doc_mock_{i}' for i in range(k)]
    try:
        query_embedding = model.encode(query).tolist() if model else None
        
        bm25_body = {"size": 100, "query": {"match": {"texto": {"query": query, "operator": "or"}}}}
        bm25_resp = client.search(index=index_name, body=bm25_body)
        bm25_scores = {hit['_id']: hit['_score'] for hit in bm25_resp['hits']['hits']}
        
        knn_body = {"size": 100, "query": {"knn": {"embedding": {"vector": query_embedding, "k": 100}}}}
        knn_resp = client.search(index=index_name, body=knn_body)
        knn_scores = {hit['_id']: hit['_score'] for hit in knn_resp['hits']['hits']}
        
        bm25_values = list(bm25_scores.values())
        knn_values = list(knn_scores.values())
        
        bm25_min, bm25_max = min(bm25_values), max(bm25_values)
        knn_min, knn_max = min(knn_values), max(knn_values)
        
        all_docs = set(bm25_scores.keys()) | set(knn_scores.keys())
        final_scores = {}
        
        for doc_id in all_docs:
            norm_bm25 = (bm25_scores.get(doc_id, 0) - bm25_min) / (bm25_max - bm25_min) if bm25_max > bm25_min else 0
            norm_knn = (knn_scores.get(doc_id, 0) - knn_min) / (knn_max - knn_min) if knn_max > knn_min else 0
            final_scores[doc_id] = alpha * norm_bm25 + (1 - alpha) * norm_knn
        
        sorted_docs = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
        return [doc_id for doc_id, _ in sorted_docs[:k]]
    except Exception as e:
        print(f"Erro em Hybrid MinMax: {e}")
        return []


print("Funções de busca definidas.")

## 7. Executar Avaliação Completa

In [ ]:
evaluation_results = []

for query_obj in queries_avaliacao:
    query_id = query_obj['query_id']
    query_text = query_obj['texto']
    relevant_docs = set(query_obj['documentos_relevantes'])
    
    bm25_results = search_bm25(client, query_text, k=10)
    knn_results = search_knn(client, query_text, model, k=10)
    hybrid_rrf_results = search_hybrid_rrf(client, query_text, model, k=10)
    hybrid_minmax_results = search_hybrid_minmax(client, query_text, model, k=10)
    
    for mode_name, results in [('BM25', bm25_results), ('KNN', knn_results), ('Hybrid_RRF', hybrid_rrf_results), ('Hybrid_MinMax', hybrid_minmax_results)]:
        mrr = calc_mrr(results, relevant_docs, k=10)
        recall = calc_recall_at_k(results, relevant_docs, k=10)
        ndcg = calc_ndcg_at_k(results, relevant_docs, k=10)
        
        evaluation_results.append({
            'Query_ID': query_id,
            'Modo': mode_name,
            'MRR@10': mrr,
            'Recall@10': recall,
            'NDCG@10': ndcg
        })

df_results = pd.DataFrame(evaluation_results)
print(f"Avaliação completa: {len(df_results)} combinações query-modo")
print("\nPrimeiras 8 linhas:")
print(df_results.head(8))

## 8. Estatísticas Agregadas por Modo

In [ ]:
modo_stats = df_results.groupby('Modo')[['MRR@10', 'Recall@10', 'NDCG@10']].agg(['mean', 'std', 'min', 'max'])
print("\n=== ESTATÍSTICAS AGREGADAS POR MODO ===")
print(modo_stats.round(4))

print("\n=== MRR@10 MÉDIO ===")
mrr_by_mode = df_results.groupby('Modo')['MRR@10'].mean().sort_values(ascending=False)
print(mrr_by_mode.round(4))

print("\n=== RECALL@10 MÉDIO ===")
recall_by_mode = df_results.groupby('Modo')['Recall@10'].mean().sort_values(ascending=False)
print(recall_by_mode.round(4))

print("\n=== NDCG@10 MÉDIO ===")
ndcg_by_mode = df_results.groupby('Modo')['NDCG@10'].mean().sort_values(ascending=False)
print(ndcg_by_mode.round(4))

## 9. Visualizações

### Gráfico 1: Barras Agrupadas - MRR por Modo

In [ ]:
mrr_data = df_results.groupby('Modo')['MRR@10'].apply(list).to_dict()

fig, ax = plt.subplots(figsize=(10, 6))
modes = list(mrr_data.keys())
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

x_pos = np.arange(len(modes))
mrr_means = [np.mean(mrr_data[mode]) for mode in modes]
mrr_stds = [np.std(mrr_data[mode]) for mode in modes]

bars = ax.bar(x_pos, mrr_means, yerr=mrr_stds, capsize=5, alpha=0.8, color=colors, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Modo de Busca', fontsize=12, fontweight='bold')
ax.set_ylabel('MRR@10', fontsize=12, fontweight='bold')
ax.set_title('Mean Reciprocal Rank (MRR@10) por Modo de Busca', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(modes, fontsize=11)
ax.set_ylim(0, max(mrr_means) * 1.3)
ax.grid(axis='y', alpha=0.3)

for i, (bar, mean) in enumerate(zip(bars, mrr_means)):
    ax.text(bar.get_x() + bar.get_width()/2, mean + mrr_stds[i] + 0.02, f'{mean:.3f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('mrr_por_modo.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico 1 salvo como 'mrr_por_modo.png'")

### Gráfico 2: Heatmap - Query × Modo para Recall@10

In [ ]:
pivot_recall = df_results.pivot_table(values='Recall@10', index='Query_ID', columns='Modo')

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(pivot_recall, annot=True, fmt='.2f', cmap='YlGnBu', cbar_kws={'label': 'Recall@10'},
            ax=ax, linewidths=0.5, linecolor='gray')

ax.set_title('Recall@10 por Query e Modo de Busca', fontsize=13, fontweight='bold', pad=20)
ax.set_xlabel('Modo de Busca', fontsize=12, fontweight='bold')
ax.set_ylabel('Query ID', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('recall_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico 2 salvo como 'recall_heatmap.png'")

### Gráfico 3: Scatter - MRR vs Recall para os 4 Modos

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

modes = df_results['Modo'].unique()
colors_scatter = {'BM25': '#1f77b4', 'KNN': '#ff7f0e', 'Hybrid_RRF': '#2ca02c', 'Hybrid_MinMax': '#d62728'}

for mode in modes:
    mode_data = df_results[df_results['Modo'] == mode]
    ax.scatter(mode_data['MRR@10'], mode_data['Recall@10'], 
              label=mode, s=150, alpha=0.7, edgecolors='black', linewidth=1.5,
              color=colors_scatter.get(mode, 'gray'))

ax.set_xlabel('MRR@10', fontsize=12, fontweight='bold')
ax.set_ylabel('Recall@10', fontsize=12, fontweight='bold')
ax.set_title('Relação entre MRR@10 e Recall@10 por Modo de Busca', fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig('mrr_vs_recall_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico 3 salvo como 'mrr_vs_recall_scatter.png'")

## 10. Análise Qualitativa: 3 Casos de Uso

In [ ]:
selected_queries = ['q1', 'q2', 'q3']
query_texts = {q['query_id']: q['texto'] for q in queries_avaliacao}

print("=" * 100)
print("ANÁLISE QUALITATIVA: TOP-3 RESULTADOS POR MODO E QUERY")
print("=" * 100)

for query_id in selected_queries:
    query_text = query_texts[query_id]
    print(f"\n### QUERY {query_id}: {query_text} ###\n")
    
    query_results = df_results[df_results['Query_ID'] == query_id].copy()
    
    for mode in ['BM25', 'KNN', 'Hybrid_RRF', 'Hybrid_MinMax']:
        mode_row = query_results[query_results['Modo'] == mode].iloc[0]
        print(f"{mode:15} | MRR: {mode_row['MRR@10']:.4f} | Recall: {mode_row['Recall@10']:.4f} | NDCG: {mode_row['NDCG@10']:.4f}")
    print("-" * 100)

## 11. Conclusões e Insights

### Template para Análise do Aluno

#### 1. Qual modo apresentou melhor desempenho geral? Por quê?

*[Espaço para resposta]*

#### 2. Em quais tipos de queries o KNN superou o BM25?

*[Espaço para resposta]*

#### 3. A busca híbrida agregou valor?

*[Espaço para resposta]*

#### 4. Qual estratégia recomendaria para um sistema jurídico?

*[Espaço para resposta]*

#### 5. Qual métrica (MRR, Recall, NDCG) é mais relevante para aplicações jurídicas?

*[Espaço para resposta]*

## Referências (ABNT)

BAAI. **Bge-m3: Multilingual sentence embeddings model**. GitHub: BAAI/bge-m3, 2024. Disponível em: https://github.com/FlagOpen/FlagEmbedding. Acesso em: 2026.

OPENSEARCH. **OpenSearch documentation**. Disponível em: https://opensearch.org/docs/. Acesso em: 2026.

JÄRVELIN, K.; KEKÄLÄINEN, J. **IR evaluation with asymmetrical relevance judgements**. In: Proceedings of the 25th Annual International ACM SIGIR Conference on Research and Development in Information Retrieval. New York: ACM, 2002. p. 315-322.

ROBERTSON, S. **Understanding Inverse Document Frequency: On theoretical arguments for IDF**. Journal of Documentation, v. 60, n. 5, p. 503-520, 2004.

CRASWELL, N.; MITRA, B.; YILMAZ, E.; CAMPOS, D. **MS MARCO: A Human Generated MAchine Reading COmprehension Dataset**. arXiv preprint arXiv:1611.09268, 2016.